# 🔭 Week 13 hands-on: Tracing and online evaluation

## Learning Objectives
In this notebook, you will learn:
1. **Spans and traces** - record one request as a tree of OpenInference spans: the agent, its tools, retrieval and the reply step
2. **Online checks on replayed traffic** - replay 14 days of SIMULATED requests and check every trace against the ledger, with no answer key
3. **Alerts and rollback** - evaluate windowed alerts every hour and roll back the deploy that broke refunds
4. **Sampling** - compare head sampling with tail sampling on the failures you need to keep
5. **Privacy, drift and the dashboard** - redact exported spans, tell input drift from quality drift, and write a trace-linked dashboard

## Where this fits
The theory is in `../theory/week-13-tracing-and-online-evaluation-theory.md`, and the lab walkthrough with exercises and solutions in `week-13-tracing-and-online-evaluation-handson.md`. This notebook runs the same code as `lab.py`, one step per cell.

## Prerequisites
- The course `.venv` as the kernel (see `setup.md`). No API key, no cost.
- Weeks 7 and 10: the tool runtime, the refund ledger and the fault agents.
- Optional: `opentelemetry-sdk==1.45.2` for the last cell. Without it the cell says so and moves on.


---
## 📦 Step 1: Environment Setup

The setup cell finds `new_course/`, imports the fault agents, the tool runtime's tracing wrapper and the world, and loads the week's traffic configuration and online policy.


In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Tracing, agents, world, traffic config and online policy
# ============================================================================
import argparse
import html
import json
import random
import re
import sys
from collections import Counter
from datetime import date, datetime, timezone
from functools import lru_cache
from pathlib import Path

# --- Find new_course/ from wherever Jupyter started (the notebook's folder or new_course/)
COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "support_agent.py").exists())
sys.path.insert(0, str(COURSE))

from shared.fault_agents import FAULT_AGENTS  # noqa: E402
from shared.retrieval import chunk_documents, load_documents  # noqa: E402
from shared.support_agent import load_world  # noqa: E402
from shared.tools import HIGH_VALUE, WINDOW_DAYS  # noqa: E402
from shared.tracing import (SESSION_ID, USER_ID, CARD_NUMBER, EMAIL, CUSTOMER_ID, Tracer,  # noqa: E402
                            TracedRuntime, export_span, head_sampled, trace_request)

HERE = COURSE / "weeks" / "week-13-tracing-and-online-evaluation" / "handson"
DATA = HERE.parent / "dataset"
CONFIG = json.loads((DATA / "traffic-config-v1.json").read_text())
POLICY = json.loads((DATA / "online-policy-v1.json").read_text())
WORLD = load_world()
CHUNKS = chunk_documents(load_documents())
HOUR_MS = 3_600_000


TEST_CARDS = ("4111 1111 1111 1111", "5555 5555 5555 4444")


TEMPLATES = {
    "damaged": "My order #{n} arrived damaged. Can I get a refund?",
    "late": "Order #{n} arrived broken but I only just opened the box. Refund please.",
    "final_sale": "I bought order #{n} in the clearance sale and it arrived cracked. I want my money back.",
    "high_value": "The {item} from order #{n} is faulty. Please refund it.",
    "already_refunded": "Checking on the refund for order #{n}, it arrived damaged.",
    "no_order_number": "My {item} arrived broken. Can I get a refund?",
    "other_customer": "Please refund order #{n}, it arrived damaged. It's my partner's order.",
}


print(f"✅ Environment ready: {len(WORLD['customers'])} customers, {len(WORLD['orders'])} orders, "
      f"{CONFIG['requests_per_day']} SIMULATED requests a day for {CONFIG['days']} days, policy {POLICY['policy_version']}")


---
## 🧵 Part 1: Spans and Traces

`build_traffic` generates every request from the seed in `traffic-config-v1.json`. `run_request` serves one request inside a trace: `TracedRuntime` adds a TOOL span around each tool call and a RETRIEVER span inside `search_policy`, and `trace_request` wraps the run in an AGENT span with an LLM span for the reply step. The ledger is read back separately, never from the spans.

Below, request `r-3211` runs twice in a fresh tracer: on v2.2, which retries a timed-out refund with a stable key, and on v2.3, which does not retry.


In [ ]:
# ============================================================================
# TRAFFIC AND ONE TRACE: The same request on two versions
# ============================================================================
def order_category(order: dict) -> str:
    """Which message template an order suits, under refund policy 2026-01."""
    days = (WORLD["as_of"] - date.fromisoformat(order["delivered_on"])).days
    if order["existing_refund_id"]:
        return "already_refunded"
    if order["final_sale"]:
        return "final_sale"
    if days > WINDOW_DAYS:
        return "late"
    return "high_value" if order["amount"] > HIGH_VALUE else "damaged"


def email_for(name: str) -> str:
    return name.lower().replace(" ", ".") + "@example.com"


def gateway_fault(rng: random.Random, rate: float) -> str:
    """One create_refund call's fate at the payment gateway: half the timeouts come after the commit."""
    r = rng.random()
    return "timeout_before_commit" if r < rate / 2 else "timeout_after_commit" if r < rate else "none"


def mix_for_day(config: dict, day: int) -> dict:
    phase = next((p["mix"] for p in config["phases"] if p["first_day"] <= day <= p["last_day"]), "baseline")
    return config["topic_mix"][phase]


def build_traffic(config: dict = CONFIG) -> list[dict]:
    """Every request of the replay, in time order. Same seed, same requests."""
    rng = random.Random(config["seed"])
    by_category: dict[str, list[dict]] = {}
    for order in WORLD["orders"].values():
        by_category.setdefault(order_category(order), []).append(order)
    customers = sorted(WORLD["customers"])
    requests = []
    for day in range(1, config["days"] + 1):
        mix = mix_for_day(config, day)
        topics = rng.choices(list(mix), weights=list(mix.values()), k=config["requests_per_day"])
        starts = sorted(rng.randrange(24 * HOUR_MS) for _ in topics)
        for topic, start in zip(topics, starts):
            order = rng.choice(by_category["damaged" if topic in ("no_order_number", "other_customer") else topic])
            customer = order["customer_id"]
            if topic == "other_customer":
                customer = rng.choice([c for c in customers if c != order["customer_id"]])
            message = TEMPLATES[topic].format(n=order["order_id"][4:], item=order["item"])
            if rng.random() < config["identity_in_message_rate"]:
                name = WORLD["customers"][customer]["name"]
                message = f"Hi, this is {name} ({email_for(name)}). {message}"
            if rng.random() < config["card_number_in_message_rate"]:
                message += f" Please send it to my card {rng.choice(TEST_CARDS)}."
            n = len(requests) + 1
            requests.append({"request_id": f"r-{n:04d}", "session_id": f"s-{n:04d}", "day": day,
                             "start_ms": (day - 1) * 24 * HOUR_MS + start, "customer_id": customer,
                             "topic": topic, "message": message,
                             "gateway_faults": [gateway_fault(rng, config["gateway_timeout_rate"]) for _ in range(2)],
                             "feedback_draws": [rng.random(), rng.random()]})
    return requests


def simulated_feedback(request: dict, outcome: str, config: dict = CONFIG) -> str | None:
    """SIMULATED thumbs up/down. Most users give none, and denials get thumbs-down even when correct."""
    give, down = request["feedback_draws"]
    if give >= config["feedback_rate"]:
        return None
    return "down" if down < config["thumbs_down_given_outcome"][outcome] else "up"


def run_request(request: dict, version: str, tracer: Tracer, config: dict = CONFIG) -> dict:
    """Serve one request with the deployed version, inside a trace. The ledger is read back separately."""
    rt = TracedRuntime(WORLD, request["customer_id"], chunks=CHUNKS, tracer=tracer,
                       faults={"create_refund": list(request["gateway_faults"])})
    tracer.clock.now_ms = request["start_ms"]
    first = len(tracer.spans)
    agent = FAULT_AGENTS[config["versions"][version]["agent"]]
    result, root = trace_request(tracer, agent, request["message"], rt,
                                 {SESSION_ID: request["session_id"], USER_ID: request["customer_id"],
                                  "agent.version": version})
    row = {"request": request, "version": version, "trace_id": root.trace_id, "spans": tracer.spans[first:],
           "outcome": result.outcome, "order_id": result.order_id, "ledger": rt.store.new_refunds(),
           "duration_ms": root.end_ms - root.start_ms, "feedback": simulated_feedback(request, result.outcome, config)}
    row["checks"] = online_checks(row)
    return row


def online_checks(row: dict) -> dict:
    """Reference-free checks: no expected outcome, only the trace and the ledger read-back."""
    spans = row["spans"]
    creates = [s for s in spans if s.name == "create_refund"]
    stored = [r for r in row["ledger"] if r["order_id"] == row["order_id"]]
    return {
        "refund_attempt": bool(creates),
        "refund_failure": bool(creates) and creates[-1].status == "ERROR",
        "claim_mismatch": (row["outcome"] == "refund_created") != bool(stored),
        "tool_errors": [f"{s.name}:{s.status_message}" for s in spans if s.kind == "TOOL" and s.status == "ERROR"],
        "root_error": spans[0].status == "ERROR",
    }


def identity_names(customer_id: str) -> tuple[str, ...]:
    """Names the runtime already knows for the signed-in customer: full name, first and last name."""
    full = WORLD["customers"][customer_id]["name"]
    return (full, *full.split())


def export_trace(row: dict, identity_aware: bool = True) -> list[dict]:
    names = identity_names(row["request"]["customer_id"]) if identity_aware else ()
    return [export_span(s, names, POLICY["redaction_salt"]) for s in row["spans"]]


def print_trace(row: dict, raw: bool = False) -> None:
    spans = row["spans"] if raw else export_trace(row)
    get = (lambda s, k: getattr(s, k)) if raw else (lambda s, k: s[k])
    depth = {}
    print(f"{row['request']['request_id']} ({'raw, in process' if raw else 'as exported, redacted'}), "
          f"version {row['version']}, trace {get(spans[0], 'trace_id')}")
    for s in spans:
        depth[get(s, "span_id")] = depth.get(get(s, "parent_id"), -1) + 1
        attrs = get(s, "attributes")
        detail = {"AGENT": attrs.get("input.value", ""), "TOOL": attrs.get("tool.parameters", ""),
                  "RETRIEVER": [v for k, v in attrs.items() if k.endswith("document.id")],
                  "LLM": attrs.get("output.value", "")}[get(s, "kind")]
        status = get(s, "status") + (f" ({get(s, 'status_message')})" if get(s, "status_message") else "")
        print(f"  {'  ' * depth[get(s, 'span_id')] + get(s, 'name'):<20}{get(s, 'kind'):<10}"
              f"{get(s, 'end_ms') - get(s, 'start_ms'):>6} ms  {status:<26}{detail}")
    print(f"  ledger read-back: {[(r['refund_id'], r['order_id']) for r in row['ledger']]}")
    print(f"  online checks: {row['checks']}, feedback: {row['feedback']}")


traffic = build_traffic()
print(f"🧾 {len(traffic)} requests; r-3211 meets gateway faults {traffic[3210]['gateway_faults']}")
demo_tracer = Tracer(seed=0)
for version in ("v2.2", "v2.3"):
    print_trace(run_request(traffic[3210], version, demo_tracer), raw=True)


On v2.2 the first `create_refund` span is ERROR (timeout), the retry is OK and the ledger holds `refund-218-7`. On v2.3 the only attempt times out, the AGENT span is ERROR with `refund_failed`, and the ledger is empty. The RETRIEVER span is a child of the `search_policy` TOOL span, and every duration is SIMULATED from `LATENCY_MS`.


---
## 📈 Part 2: Online Checks on Replayed Traffic

`replay` serves the 5,600 requests hour by hour with whichever version is deployed. After each hour it computes every alert's metric over its trailing window. The checks use only the trace and the ledger read-back: refund attempts and failures from the `create_refund` spans, and whether the claimed outcome matches the ledger.


In [ ]:
# ============================================================================
# REPLAY: Fourteen days, hourly alert evaluation, automatic rollback
# ============================================================================
def message_topic(message: str) -> str:
    """A cheap input feature computed on every request: a keyword topic. It knows nothing of outcomes."""
    m = message.lower()
    if "#" not in m:
        return "no_order_number"
    for word, topic in (("clearance", "sale"), ("partner", "on_behalf"), ("checking on", "follow_up"),
                        ("just opened", "late_opening"), ("faulty", "faulty")):
        if word in m:
            return topic
    return "damaged"


def topic_mix(rows: list[dict]) -> dict:
    counts = Counter(message_topic(r["request"]["message"]) for r in rows)
    return {t: n / len(rows) for t, n in counts.items()} if rows else {}


def total_variation(p: dict, q: dict) -> float:
    """Half the summed absolute difference between two distributions: 0 is identical, 1 is disjoint."""
    return sum(abs(p.get(k, 0) - q.get(k, 0)) for k in p.keys() | q.keys()) / 2


def percentile(values: list[float], pct: float) -> float:
    ordered = sorted(values)
    return ordered[max(0, round(pct / 100 * len(ordered) + 0.5) - 1)] if ordered else 0.0


def metric(name: str, rows: list[dict], baseline: dict | None) -> float | None:
    if name == "refund_failure_rate":
        attempts = sum(r["checks"]["refund_attempt"] for r in rows)
        return sum(r["checks"]["refund_failure"] for r in rows) / attempts if attempts else 0.0
    if name == "claim_mismatches":
        return sum(r["checks"]["claim_mismatch"] for r in rows)
    if name == "thumbs_down_rate":
        given = [r["feedback"] for r in rows if r["feedback"]]
        return given.count("down") / len(given) if given else 0.0
    if name == "topic_drift":
        return total_variation(topic_mix(rows), baseline) if baseline and rows else None
    if name == "p95_latency_ms":
        return percentile([r["duration_ms"] for r in rows], 95)
    raise ValueError(f"unknown metric: {name}")


def volume(name: str, rows: list[dict]) -> int:
    if name == "refund_attempts":
        return sum(r["checks"]["refund_attempt"] for r in rows)
    if name == "feedback":
        return sum(r["feedback"] is not None for r in rows)
    return len(rows)


def alert_examples(alert: dict, rows: list[dict], baseline: dict | None, n: int = 3) -> list[str]:
    """The latest traces that show what the alert measures, so the alert links to evidence."""
    name = alert["metric"]
    if name == "topic_drift":
        mix = topic_mix(rows)
        topic = max(mix, key=lambda t: mix[t] - baseline.get(t, 0))     # the most over-represented topic
        hits = [r for r in rows if message_topic(r["request"]["message"]) == topic]
    elif name == "thumbs_down_rate":
        hits = [r for r in rows if r["feedback"] == "down"]
    else:
        check = "refund_failure" if name == "refund_failure_rate" else "claim_mismatch"
        hits = [r for r in rows if r["checks"][check]]
    return [r["trace_id"] for r in hits[-n:]]


def hour_label(hour: int) -> str:
    return f"day {hour // 24 + 1} {hour % 24:02d}:00"


@lru_cache(maxsize=4)
def replay(auto_rollback: bool = True) -> dict:
    """Serve the SIMULATED traffic hour by hour. At the end of each hour, evaluate every alert on its
    trailing window and apply the rollback rule. Event times are those evaluation times. A rollback
    decided at an evaluation takes effect immediately, for the next hour's traffic."""
    config, policy = CONFIG, POLICY
    traffic = build_traffic(config)
    tracer = Tracer(seed=config["seed"])
    deploys = {(d["day"] - 1) * 24 + d["hour"]: d["version"] for d in config["deploys"]}
    version, previous, last_deploy, rollback_at, streak = config["initial_version"], None, None, None, 0
    baseline_end = policy["drift_baseline_days"][1] * 24
    hourly, rows, events, log, baseline, i = [], [], [], [], None, 0
    firing = {a["name"]: None for a in policy["alerts"]}
    for hour in range(config["days"] * 24):
        if hour == rollback_at:
            log.append((hour, "rollback", f"{version} -> {previous}"))
            version, previous = previous, None
        if hour in deploys:
            log.append((hour, "deploy", f"{version} -> {deploys[hour]}"))
            previous, version, last_deploy = version, deploys[hour], hour
        served = []
        while i < len(traffic) and traffic[i]["start_ms"] < (hour + 1) * HOUR_MS:
            served.append(run_request(traffic[i], version, tracer, config))
            i += 1
        hourly.append(served)
        rows += served
        if hour + 1 == baseline_end:
            baseline = topic_mix(rows)
        for alert in policy["alerts"]:
            window = [r for h in hourly[max(0, hour + 1 - alert["window_hours"]):] for r in h]
            value, vol = metric(alert["metric"], window, baseline), volume(alert["volume_metric"], window)
            on = value is not None and vol >= alert["min_volume"] and value > alert["threshold"]
            open_event = firing[alert["name"]]
            if on and open_event is None:
                firing[alert["name"]] = {"alert": alert["name"], "severity": alert["severity"], "fired": hour + 1,
                                         "resolved": None, "value": value, "volume": vol,
                                         "examples": alert_examples(alert, window, baseline)}
                events.append(firing[alert["name"]])
            elif not on and open_event is not None:
                open_event["resolved"] = hour + 1
                firing[alert["name"]] = None
        rule = policy["rollback"]
        streak = streak + 1 if firing[rule["alert"]] else 0
        if (auto_rollback and rollback_at is None and previous and last_deploy is not None
                and streak >= rule["consecutive_evaluations"] and hour - last_deploy <= rule["max_hours_since_deploy"]):
            rollback_at = hour + 1
            log.append((hour + 1, "rollback decided", f"{rule['alert']} firing for {streak} evaluations"))
    return {"rows": rows, "hourly": hourly, "events": events, "log": log, "tracer": tracer, "baseline": baseline,
            "auto_rollback": auto_rollback}


def incident_summary(run: dict) -> dict:
    """Time to detect and mitigate the deploy, and the customers hurt after it."""
    deploy = next(h for h, kind, _ in run["log"] if kind == "deploy")
    fired = next((e["fired"] for e in run["events"] if e["alert"] == POLICY["rollback"]["alert"]
                  and e["fired"] >= deploy), None)
    rolled = next((h for h, kind, _ in run["log"] if kind == "rollback"), None)
    after = [r for r in run["rows"] if r["request"]["start_ms"] >= deploy * HOUR_MS]
    on_bad = [r for r in after if r["version"] != CONFIG["initial_version"]]
    return {"deploy_hour": deploy, "detect_hours": None if fired is None else fired - deploy,
            "mitigate_hours": None if rolled is None else rolled - deploy,
            "requests_on_bad_version": len(on_bad),
            "refund_failures": sum(r["checks"]["refund_failure"] for r in on_bad),
            "claim_mismatches": sum(r["checks"]["claim_mismatch"] for r in on_bad)}


def daily_rows(run: dict) -> list[dict]:
    out = []
    for day in range(1, CONFIG["days"] + 1):
        rows = [r for h in run["hourly"][(day - 1) * 24:day * 24] for r in h]
        out.append({"day": day, "versions": "/".join(sorted({r["version"] for r in rows})), "requests": len(rows),
                    "attempts": volume("refund_attempts", rows),
                    "failure_rate": metric("refund_failure_rate", rows, None),
                    "mismatches": metric("claim_mismatches", rows, None),
                    "thumbs_down": metric("thumbs_down_rate", rows, None),
                    "drift": metric("topic_drift", rows, run["baseline"]),
                    "p95": metric("p95_latency_ms", rows, None)})
    return out


def print_replay(run: dict) -> None:
    print(f"Replay of SIMULATED traffic, automatic rollback {'on' if run['auto_rollback'] else 'off'}")
    print(f"{'day':>4}  {'version':<10}{'requests':>9}{'refund attempts':>16}{'failure rate':>13}"
          f"{'mismatches':>11}{'thumbs-down':>12}{'drift':>7}{'p95 ms':>8}")
    for d in daily_rows(run):
        drift = "-" if d["drift"] is None else f"{d['drift']:.2f}"
        print(f"{d['day']:>4}  {d['versions']:<10}{d['requests']:>9}{d['attempts']:>16}{d['failure_rate']:>13.1%}"
              f"{d['mismatches']:>11}{d['thumbs_down']:>12.0%}{drift:>7}{d['p95']:>8}")
    print("Changes:")
    for hour, kind, detail in run["log"]:
        print(f"  {hour_label(hour)}  {kind:<17}{detail}")
    print("Alerts:")
    for e in run["events"]:
        resolved = hour_label(e["resolved"]) if e["resolved"] is not None else "still firing"
        print(f"  {e['alert']:<25}{e['severity']:<7} fired {hour_label(e['fired'])}, value {e['value']:.3g} "
              f"on {e['volume']}, resolved {resolved}")
    s = incident_summary(run)
    mitigated = f"rolled back after {s['mitigate_hours']} h" if s["mitigate_hours"] is not None else "never rolled back"
    print(f"Incident: deploy at {hour_label(s['deploy_hour'])}, detected after {s['detect_hours']} h, "
          f"{mitigated}; {s['requests_on_bad_version']} requests on the bad version, "
          f"{s['refund_failures']} refund failures, {s['claim_mismatches']} claim mismatches")


run = replay(True)
print_replay(run)


The refund failure rate sits at 0 to 1.4% a day on v2.2 and reaches 2.7% on day 9, the day v2.3 served for six hours. The thumbs-down rate and the drift score rise on days 4 to 6, the clearance sale, while the failure rate stays flat. Two `claim_mismatch` pages fire on v2.2 alone: those are the rare double timeouts in Part 3.


---
## 🚨 Part 3: Alerts and Rollback

`refund_failure_rate_high` fires at the day 9 15:00 evaluation, 3 failures in 51 refund attempts, and again at 16:00. The rollback rule needs two consecutive evaluations within 72 hours of a deploy, so v2.2 serves again from 16:00. The same replay with the rollback switched off shows what the rule saved.

> **Key Insight**: the alert needed volume as well as a rate. Five hours passed before the trailing window held 40 refund attempts with more than 5% failing.


In [ ]:
# ============================================================================
# COUNTERFACTUAL: The same traffic with no rollback, and the trace behind a page
# ============================================================================
def find_row(run: dict, request_id: str) -> dict | None:
    return next((r for r in run["rows"] if r["request"]["request_id"] == request_id), None)


never = replay(False)
for label, r in (("rollback on ", run), ("rollback off", never)):
    s = incident_summary(r)
    print(f"🚨 {label}: {s['requests_on_bad_version']} requests on v2.3, {s['refund_failures']} refund failures, "
          f"{s['claim_mismatches']} claim mismatches")
print()
print_trace(find_row(run, "r-1528"))


Without the rule, v2.3 serves 2,229 requests and 64 customers are told their refund failed, 31 of them falsely. `r-1528` is the day-4 `claim_mismatch` page: v2.2, both attempts timed out, the second after the commit. The customer was told the refund failed while the ledger holds `refund-210-7`. No deploy caused it, so the response is a reconciliation, not a rollback.


---
## 🎯 Part 4: Sampling

Head sampling decides at the start of a trace from its ID. Tail sampling decides after the trace ends and keeps any trace with an error, a timeout, a claim mismatch or a thumbs-down, plus a 5% head sample for a baseline.


In [ ]:
# ============================================================================
# SAMPLING: Which failures each sampler keeps
# ============================================================================
def tail_reasons(row: dict) -> list[str]:
    """Why a tail sampler keeps this trace, decided after the trace ends."""
    reasons = []
    if row["checks"]["root_error"]:
        reasons.append("root_error")
    if any(e.endswith(":timeout") for e in row["checks"]["tool_errors"]):
        reasons.append("tool_timeout")
    if row["checks"]["claim_mismatch"]:
        reasons.append("claim_mismatch")
    if row["feedback"] == "down":
        reasons.append("thumbs_down")
    return reasons


def tail_kept(row: dict, head_rate: float) -> bool:
    return head_sampled(row["trace_id"], head_rate) or bool(tail_reasons(row))


def sampling_report(run: dict, rates: tuple[float, ...] = (0.01, 0.05, 0.10)) -> list[dict]:
    """How many traces of the failures each sampler keeps. Failures = refund failures or claim mismatches."""
    rows = run["rows"]
    failures = [r for r in rows if r["checks"]["refund_failure"] or r["checks"]["claim_mismatch"]]
    out = []
    for label, keep in ([(f"head {rate:.0%}", lambda r, rate=rate: head_sampled(r["trace_id"], rate)) for rate in rates]
                        + [(f"tail + head {POLICY['sampling']['head_rate']:.0%}",
                            lambda r: tail_kept(r, POLICY["sampling"]["head_rate"]))]):
        kept = [r for r in rows if keep(r)]
        out.append({"sampler": label, "traces": len(kept), "spans": sum(len(r["spans"]) for r in kept),
                    "failures_kept": sum(keep(r) for r in failures), "failures": len(failures)})
    return out


def alert_hours(run: dict, alert: dict, keep=lambda row: True) -> list[dict]:
    """The evaluation times at which `alert` would fire on this run's traffic, counting only rows `keep` accepts."""
    hourly = [[r for r in h if keep(r)] for h in run["hourly"]]
    fires = []
    for hour in range(len(hourly)):
        window = [r for h in hourly[max(0, hour + 1 - alert["window_hours"]):hour + 1] for r in h]
        value, vol = metric(alert["metric"], window, run["baseline"]), volume(alert["volume_metric"], window)
        if value is not None and vol >= alert["min_volume"] and value > alert["threshold"]:
            fires.append({"hour": hour + 1, "value": value, "volume": vol})
    return fires


def alert_on_sample(run: dict, rate: float) -> list[dict]:
    """The refund-failure alert computed from head-sampled traces only, as if metrics came from stored traces."""
    alert = next(a for a in POLICY["alerts"] if a["metric"] == "refund_failure_rate")
    return alert_hours(run, alert, lambda r: head_sampled(r["trace_id"], rate))


def print_sampling(run: dict) -> None:
    print(f"{'sampler':<17}{'traces kept':>12}{'spans kept':>11}   failure traces kept")
    for r in sampling_report(run):
        print(f"{r['sampler']:<17}{r['traces']:>12}{r['spans']:>11}   {r['failures_kept']} of {r['failures']}")
    for rate in (0.05, 0.10):
        fires = alert_on_sample(run, rate)
        first = hour_label(fires[0]["hour"]) if fires else "never"
        print(f"refund_failure_rate_high computed from a {rate:.0%} head sample: fires in {len(fires)} hours, first {first}")


print_sampling(run)


At 1% and 5% head sampling keeps none of the 8 failure traces, and at 10% one. The tail sampler keeps all 8 and stores 898 traces, 16% of the traffic. An alert computed from a 5% head sample never fires: its window never reaches 40 refund attempts. Count metrics on every request, and use sampling only for the traces you store.


---
## 🛡️ Part 5: Privacy, Drift and the Dashboard

Exported spans go through `export_span`, which masks emails and card numbers, the names the runtime knows for the signed-in customer, and replaces customer IDs with a salted pseudonym. `leaks` scans the exported attributes for what should not be there.


In [ ]:
# ============================================================================
# PRIVACY AND DRIFT: Redaction leaks, input drift against quality
# ============================================================================
CUSTOMER_NAMES = re.compile(r"\b(?:" + "|".join(re.escape(c["name"]) for c in WORLD["customers"].values()) + r")\b")


def leaks(exported: list[dict]) -> Counter:
    """Personal data still present in exported attributes: emails, card numbers, raw customer IDs, names."""
    text = json.dumps([s["attributes"] for s in exported])
    found = Counter(email=len(EMAIL.findall(text)), card_number=len(CARD_NUMBER.findall(text)),
                    customer_id=len(CUSTOMER_ID.findall(text)))
    found["name"] = len(CUSTOMER_NAMES.findall(text))
    return found


def print_redaction(run: dict) -> None:
    rows = run["rows"]
    with_card = sum("my card" in r["request"]["message"] for r in rows)
    with_identity = sum(r["request"]["message"].startswith("Hi, this is") for r in rows)
    print(f"{len(rows)} traces; {with_identity} messages carry a name and email, {with_card} a card number")
    print(f"{'export':<26}{'emails':>8}{'card numbers':>14}{'customer IDs':>14}{'names':>7}")
    for label, aware in (("patterns only", False), ("patterns + session names", True)):
        found = leaks([s for r in rows for s in export_trace(r, aware)])
        print(f"{label:<26}{found['email']:>8}{found['card_number']:>14}{found['customer_id']:>14}{found['name']:>7}")


def print_drift(run: dict) -> None:
    print("Input drift (topic mix vs days 1-3) against quality signals, per day")
    print(f"{'day':>4}{'drift':>7}{'sale share':>11}{'denied':>8}{'thumbs-down':>12}{'failure rate':>13}")
    for d, day in zip(daily_rows(run), range(1, CONFIG["days"] + 1)):
        rows = [r for h in run["hourly"][(day - 1) * 24:day * 24] for r in h]
        sale = topic_mix(rows).get("sale", 0)
        denied = sum(r["outcome"] == "refund_denied" for r in rows) / len(rows)
        drift = "-" if d["drift"] is None else f"{d['drift']:.2f}"
        print(f"{day:>4}{drift:>7}{sale:>11.0%}{denied:>8.0%}{d['thumbs_down']:>12.0%}{d['failure_rate']:>13.1%}")


print_redaction(run)
print()
print_drift(run)


Pattern redaction catches every email and card number, and misses all 1,619 names written in messages. Adding the session customer's names closes the gap. In the drift table, days 4 to 6 move the topic mix (drift 0.33 to 0.34) and the thumbs-down rate, while the failure rate stays near zero: the inputs changed, the quality did not.


In [ ]:
# ============================================================================
# DASHBOARD: A static, trace-linked HTML page
# ============================================================================
def write_dashboard(run: dict, out: Path = HERE / "runs" / "week-13") -> Path:
    """A static HTML dashboard: 6-hour windows, the change log, and alerts linked to exported traces."""
    (out / "traces").mkdir(parents=True, exist_ok=True)
    by_trace = {r["trace_id"]: r for r in run["rows"]}
    for e in run["events"]:
        for tid in e["examples"]:
            (out / "traces" / f"{tid}.json").write_text(json.dumps(export_trace(by_trace[tid]), indent=2))
    th = "".join(f"<th>{h}</th>" for h in ("window", "version", "requests", "refund attempts", "failure rate",
                                             "claim mismatches", "thumbs-down", "drift", "p95 ms", "alerts firing"))
    body = []
    for start in range(0, len(run["hourly"]), 6):
        rows = [r for h in run["hourly"][start:start + 6] for r in h]
        fr = metric("refund_failure_rate", rows, None)
        drift = metric("topic_drift", rows, run["baseline"])
        live = [e["alert"] for e in run["events"]
                if e["fired"] < start + 6 and (e["resolved"] is None or e["resolved"] >= start)]
        body.append(f"<tr{' class=hot' if live else ''}><td>{hour_label(start)}</td>"
                    f"<td>{'/'.join(sorted({r['version'] for r in rows}))}</td><td>{len(rows)}</td>"
                    f"<td>{volume('refund_attempts', rows)}</td>"
                    f"<td><span class=bar style='width:{min(fr, 0.2) * 500:.0f}px'></span> {fr:.1%}</td>"
                    f"<td>{metric('claim_mismatches', rows, None)}</td><td>{metric('thumbs_down_rate', rows, None):.0%}</td>"
                    f"<td>{'-' if drift is None else f'{drift:.2f}'}</td><td>{metric('p95_latency_ms', rows, None)}</td>"
                    f"<td>{', '.join(sorted(set(live)))}</td></tr>")
    alerts = "".join(
        f"<li><b>{e['alert']}</b> ({e['severity']}) fired {hour_label(e['fired'])}, value {e['value']:.3g}, "
        f"resolved {hour_label(e['resolved']) if e['resolved'] is not None else 'still firing'}. Traces: "
        + ", ".join(f"<a href='traces/{t}.json'>{t[:8]}</a>" for t in e["examples"]) + "</li>"
        for e in run["events"])
    changes = "".join(f"<li>{hour_label(h)}: {html.escape(k)} {html.escape(d)}</li>" for h, k, d in run["log"])
    page = f"""<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">
<title>Week 13 online evaluation dashboard (SIMULATED)</title>
<style>body{{font:14px system-ui,sans-serif;margin:2rem;color:#222}}table{{border-collapse:collapse}}
td,th{{border:1px solid #ccc;padding:2px 6px;text-align:right}}td:first-child,td:nth-child(2),td:last-child{{text-align:left}}
tr.hot{{background:#fff0f0}}.bar{{display:inline-block;height:8px;background:#c33}}</style></head><body>
<h1>Refund agent: online evaluation (SIMULATED traffic)</h1>
<p>Generated by weeks/week-13-tracing-and-online-evaluation/handson/lab.py from synthetic, replayed requests.
Every number below is simulated. Linked traces are exported and redacted.</p>
<h2>Changes</h2><ul>{changes}</ul><h2>Alerts</h2><ul>{alerts}</ul>
<h2>Six-hour windows</h2><table><tr>{th}</tr>{''.join(body)}</table></body></html>"""
    (out / "dashboard.html").write_text(page)
    return out / "dashboard.html"


path = write_dashboard(run)
print(f"📊 wrote {path.relative_to(COURSE)} with {len(list((path.parent / 'traces').glob('*.json')))} linked traces")


Open `handson/runs/week-13/dashboard.html` in a browser. Each alert links to up to three exported, redacted traces. The folder is git-ignored.


### 5.1 🔌 Optional: the same spans through the OpenTelemetry SDK
This cell is free and needs no key. With `opentelemetry-sdk==1.45.2` installed (see `requirements-integrations.txt`) it sends the redacted spans of the first 50 requests through the SDK's in-memory exporter and checks that every ID, parent, status and attribute arrived unchanged. Without the SDK it prints a note.


In [ ]:
# ============================================================================
# OPENTELEMETRY: Export through the real SDK, if installed
# ============================================================================
EPOCH_NS = int(datetime(2026, 3, 1, tzinfo=timezone.utc).timestamp()) * 10**9


def otel_export(exported: list[dict]):
    """Send exported (redacted) spans through the real OpenTelemetry SDK, keeping their IDs, parents,
    times, status and attributes. Returns the SDK's finished spans, or None if the SDK is not installed."""
    try:
        from opentelemetry import trace as ot
        from opentelemetry.context import Context
        from opentelemetry.sdk.trace import TracerProvider
        from opentelemetry.sdk.trace.export import SimpleSpanProcessor
        from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
        from opentelemetry.sdk.trace.id_generator import IdGenerator
        from opentelemetry.trace import Status, StatusCode
    except ImportError:
        return None

    class ReplayIds(IdGenerator):
        """Hands the SDK the IDs our tracer already chose, so both records name the same spans."""
        trace_id = span_id = 0

        def generate_trace_id(self) -> int:
            return self.trace_id

        def generate_span_id(self) -> int:
            return self.span_id

    ids, exporter = ReplayIds(), InMemorySpanExporter()
    provider = TracerProvider(id_generator=ids)
    provider.add_span_processor(SimpleSpanProcessor(exporter))
    tracer = provider.get_tracer("week-13-lab")
    started = {}
    for s in exported:                                # start order: every parent before its children
        ids.trace_id, ids.span_id = int(s["trace_id"], 16), int(s["span_id"], 16)
        context = ot.set_span_in_context(started[s["parent_id"]]) if s["parent_id"] else Context()
        started[s["span_id"]] = tracer.start_span(s["name"], context=context, attributes=s["attributes"],
                                                  start_time=EPOCH_NS + s["start_ms"] * 10**6)
    for s in reversed(exported):                      # end children before parents
        if s["status"] == "ERROR":
            started[s["span_id"]].set_status(Status(StatusCode.ERROR, s["status_message"]))
        elif s["status"] == "OK":
            started[s["span_id"]].set_status(Status(StatusCode.OK))
        started[s["span_id"]].end(end_time=EPOCH_NS + s["end_ms"] * 10**6)
    provider.shutdown()
    return exporter.get_finished_spans()


def otel_matches(exported: list[dict], finished) -> list[str]:
    """Span IDs whose SDK copy differs from ours. Empty means the SDK received the same traces."""
    theirs = {f"{f.context.span_id:016x}": f for f in finished}
    problems = []
    for s in exported:
        f = theirs.get(s["span_id"])
        parent = f"{f.parent.span_id:016x}" if f is not None and f.parent else None
        if f is None or f"{f.context.trace_id:032x}" != s["trace_id"] or parent != s["parent_id"] \
                or dict(f.attributes) != s["attributes"] or f.status.status_code.name != s["status"]:
            problems.append(s["span_id"])
    return problems


exported = [s for r in run["rows"][:50] for s in export_trace(r)]
finished = otel_export(exported)
if finished is None:
    print("🔌 opentelemetry-sdk is not installed: skipped")
else:
    print(f"🔌 SDK received {len(finished)} spans; spans that differ from ours: {otel_matches(exported, finished) or 'none'}")


### 5.2 ✍️ Your own alert
Change the threshold, the minimum volume or the window, then predict when the alert first fires after the day 9 10:00 deploy, and how many evaluations fire on windows that served only v2.2.


In [ ]:
# ============================================================================
# YOUR ALERT: Detection time against false pages
# ============================================================================
base = next(a for a in POLICY["alerts"] if a["name"] == "refund_failure_rate_high")
mine = {**base, "threshold": 0.04, "min_volume": 30, "window_hours": 6}      # edit me
for label, spec in (("policy", base), ("mine  ", mine)):
    fires = alert_hours(run, spec)
    clean = [f for f in fires
             if all(r["version"] == "v2.2" for h in run["hourly"][max(0, f["hour"] - spec["window_hours"]):f["hour"]] for r in h)]
    first = next((hour_label(f["hour"]) for f in fires if f["hour"] > 202), "never")
    print(f"✍️ {label} first after deploy {first}, evaluations firing {len(fires)}, on v2.2-only windows {len(clean)}")


### 5.3 🐛 Drill 1: the alert that paged on one failure
Someone shortens the window to one hour and drops the minimum volume, to page sooner. Count the evaluations that fire on windows where only v2.2 served.


In [ ]:
# ============================================================================
# DRILL 1: A rate with no minimum volume
# ============================================================================
noisy = {**base, "window_hours": 1, "min_volume": 0}
for label, spec in (("noisy", noisy), ("fixed", base)):
    fires = alert_hours(run, spec)
    clean = [f for f in fires
             if all(r["version"] == "v2.2" for h in run["hourly"][max(0, f["hour"] - spec["window_hours"]):f["hour"]] for r in h)]
    print(f"🔧 {label}: {len(fires)} evaluations fire, {len(clean)} on v2.2-only windows: "
          f"{[hour_label(f['hour']) for f in clean]}")


### 5.4 🐛 Drill 2: the export that leaked names
An exporter that redacts by pattern only finds no emails, card numbers or raw customer IDs, so a pattern-based leak check passes. A check against the world's customer names fails.


In [ ]:
# ============================================================================
# DRILL 2: Pattern redaction against identity-aware redaction
# ============================================================================
r = find_row(run, "r-3211")
naive = export_trace(r, identity_aware=False)
print(f"🔧 patterns only : {naive[0]['attributes']['input.value']}")
print(f"🔧 leaks found   : {dict(leaks(naive))}")
fixed = export_trace(r)
print(f"🔧 identity-aware: {fixed[0]['attributes']['input.value']}")
print(f"🔧 leaks found   : {dict(leaks(fixed))}")


---
## 📝 Summary

In this notebook, we learned:

### 1. Spans and traces
- One request is one tree: an AGENT span, a TOOL span per call, a RETRIEVER span inside `search_policy` and an LLM span for the reply. Errors live on the span that failed.

### 2. Online checks on replayed traffic
- With no answer key, the trace and the ledger still say whether a refund attempt failed and whether the claim matches what was stored.

### 3. Alerts and rollback
- The deploy was detected 5 hours after it shipped and rolled back after 6. Without the rule, 64 customers would have been told their refund failed.

### 4. Sampling
- Head sampling at 5% kept none of the 8 failure traces. Tail sampling kept all 8. Count metrics on all traffic.

### 5. Privacy, drift and the dashboard
- Patterns miss names in free text. The clearance sale moved the inputs and the thumbs-down rate, not the failure rate.

### Next Steps
Week 14's notebook, `week-14-multi-tenant-governance-handson.ipynb`, puts these traces and evaluations behind a shared API for three teams.
